# 生成实验结果GIF动画

这个notebook会为所有实验结果生成GIF动画，展示从球体到牛模型的优化过程。

## 生成的GIF包括：
1. **1_baseline_optimization.gif** - 基础版SGD优化过程
2. **2_improved_adam_optimization.gif** - 改进版Adam优化过程
3. **3_textured_v3_optimization.gif** - 三阶段纹理优化过程
4. **0_comparison.gif** - 三个版本并排对比（可选）

---

## 1. 环境检查

In [1]:
import os
import re
import torch
import numpy as np
from PIL import Image, ImageDraw, ImageFont
import imageio
from pathlib import Path

import pytorch3d
from pytorch3d.io import load_obj
from pytorch3d.structures import Meshes
from pytorch3d.renderer import (
    look_at_view_transform,
    FoVPerspectiveCameras,
    PointLights,
    RasterizationSettings,
    MeshRenderer,
    MeshRasterizer,
    SoftPhongShader,
    TexturesVertex
)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"🚀 使用设备: {device}")
print(f"📦 PyTorch3D版本: {pytorch3d.__version__}")

ModuleNotFoundError: No module named 'pytorch3d'

## 2. 配置参数

In [ ]:
# ========== 配置参数 ==========
BASE_DIR = "results"           # 结果文件夹路径
OUTPUT_DIR = "gifs"            # GIF输出路径
IMAGE_SIZE = 512               # 渲染图像大小
FPS = 10                       # GIF帧率
MAX_FRAMES = 50                # 最大帧数（太多会很大）
GENERATE_COMPARISON = True     # 是否生成对比GIF

# 创建输出目录
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"✅ 配置完成")
print(f"   结果文件夹: {BASE_DIR}")
print(f"   输出文件夹: {OUTPUT_DIR}")
print(f"   图像大小: {IMAGE_SIZE}x{IMAGE_SIZE}")
print(f"   帧率: {FPS} fps")
print(f"   最大帧数: {MAX_FRAMES}")

## 3. 工具函数

In [ ]:
def setup_renderer(image_size=512, device=device):
    """配置渲染器"""
    # 相机位置（从上方俯视）
    R, T = look_at_view_transform(dist=2.7, elev=20, azim=45)
    cameras = FoVPerspectiveCameras(device=device, R=R, T=T)
    
    # 光源
    lights = PointLights(device=device, location=[[0.0, 0.0, 3.0]])
    
    # 光栅化设置
    raster_settings = RasterizationSettings(
        image_size=image_size,
        blur_radius=0.0,
        faces_per_pixel=1,
    )
    
    # 渲染器
    renderer = MeshRenderer(
        rasterizer=MeshRasterizer(cameras=cameras, raster_settings=raster_settings),
        shader=SoftPhongShader(device=device, cameras=cameras, lights=lights)
    )
    
    return renderer


def load_and_render_mesh(obj_path, renderer, device=device, default_color=None):
    """加载并渲染单个mesh"""
    # 加载mesh
    verts, faces, aux = load_obj(obj_path)
    faces_idx = faces.verts_idx.to(device)
    verts = verts.to(device)
    
    # 归一化
    center = verts.mean(0)
    verts = verts - center
    scale = max(verts.abs().max().item(), 1e-5)
    verts = verts / scale
    
    # 处理纹理
    if aux.verts_rgb is not None and aux.verts_rgb.shape[0] > 0:
        verts_rgb = aux.verts_rgb.to(device)
    else:
        if default_color is None:
            default_color = [0.7, 0.7, 0.7]
        verts_rgb = torch.ones_like(verts) * torch.tensor(default_color, device=device)
    
    # 创建纹理和mesh
    textures = TexturesVertex(verts_features=[verts_rgb])
    mesh = Meshes(verts=[verts], faces=[faces_idx], textures=textures)
    
    # 渲染
    images = renderer(mesh)
    img = images[0, ..., :3].cpu().numpy()
    img = (img * 255).astype(np.uint8)
    
    return img


def get_obj_files(directory):
    """获取目录中所有.obj文件，按epoch排序"""
    obj_files = []
    
    for file in Path(directory).glob("*.obj"):
        # 提取epoch数字
        match = re.search(r'epoch_(\d+)', file.name)
        if match:
            epoch = int(match.group(1))
            obj_files.append((epoch, str(file)))
    
    obj_files.sort(key=lambda x: x[0])
    return obj_files


def create_gif(obj_files, output_path, renderer, fps=10, max_frames=None, default_color=None):
    """从obj文件列表创建GIF"""
    print(f"\n📹 生成GIF: {output_path}")
    print(f"   共 {len(obj_files)} 帧")
    
    frames = []
    
    # 采样帧数
    if max_frames and len(obj_files) > max_frames:
        step = len(obj_files) // max_frames
        obj_files = obj_files[::step]
        print(f"   采样至 {len(obj_files)} 帧")
    
    for i, (epoch, obj_path) in enumerate(obj_files):
        if i % max(1, len(obj_files) // 10) == 0:
            print(f"   渲染进度: {i+1}/{len(obj_files)}", end='\r')
        
        try:
            img = load_and_render_mesh(obj_path, renderer, default_color=default_color)
            
            # 添加epoch标签
            img_pil = Image.fromarray(img)
            draw = ImageDraw.Draw(img_pil)
            
            text = f"Epoch {epoch}"
            try:
                font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", 24)
            except:
                font = ImageFont.load_default()
            
            bbox = draw.textbbox((10, 10), text, font=font)
            draw.rectangle([bbox[0]-5, bbox[1]-5, bbox[2]+5, bbox[3]+5], fill=(0, 0, 0, 180))
            draw.text((10, 10), text, fill=(255, 255, 255), font=font)
            
            frames.append(np.array(img_pil))
        except Exception as e:
            print(f"\n⚠️  渲染失败: {obj_path}, 错误: {e}")
    
    print(f"\n   保存GIF...")
    imageio.mimsave(output_path, frames, fps=fps, loop=0)
    print(f"✅ 完成: {output_path}\n")

print("✅ 工具函数加载完成")

## 4. 生成各个实验的GIF

In [ ]:
print("⚙️  配置渲染器...\n")
renderer = setup_renderer(image_size=IMAGE_SIZE)

print("="*60)
print("🎬 开始生成GIF动画")
print("="*60)

# 实验配置
experiments = [
    {
        'name': 'baseline',
        'folder': os.path.join(BASE_DIR, 'baseline'),
        'output': os.path.join(OUTPUT_DIR, '1_baseline_optimization.gif'),
        'description': '基础版：SGD优化器',
        'color': [0.7, 0.7, 0.7]  # 灰色
    },
    {
        'name': 'improved_adam',
        'folder': os.path.join(BASE_DIR, 'improved_adam'),
        'output': os.path.join(OUTPUT_DIR, '2_improved_adam_optimization.gif'),
        'description': '改进版：Adam + 学习率衰减',
        'color': [0.7, 0.7, 0.7]  # 灰色
    },
    {
        'name': 'textured_v3',
        'folder': os.path.join(BASE_DIR, 'textured_v3'),
        'output': os.path.join(OUTPUT_DIR, '3_textured_v3_optimization.gif'),
        'description': '最终版：三阶段纹理优化',
        'color': None  # 使用模型自带颜色
    },
]

# 生成每个实验的GIF
for exp in experiments:
    print(f"\n📁 处理: {exp['description']}")
    
    if not os.path.exists(exp['folder']):
        print(f"⚠️  文件夹不存在: {exp['folder']}")
        continue
    
    obj_files = get_obj_files(exp['folder'])
    
    if not obj_files:
        print(f"⚠️  未找到.obj文件")
        continue
    
    create_gif(
        obj_files, 
        exp['output'], 
        renderer, 
        fps=FPS, 
        max_frames=MAX_FRAMES,
        default_color=exp['color']
    )

print("="*60)
print("🎉 所有单独的GIF生成完成！")
print("="*60)

## 5. 生成对比GIF（可选）

这个GIF会把三个版本并排显示，便于直观对比。

In [ ]:
if GENERATE_COMPARISON:
    print("\n" + "="*60)
    print("📊 生成对比GIF（三版本并排）")
    print("="*60 + "\n")
    
    comparison_size = 400  # 每个模型的显示大小
    renderer_comp = setup_renderer(image_size=comparison_size)
    
    # 获取所有实验的obj文件
    all_obj_files = []
    colors = [[0.7, 0.7, 0.7], [0.7, 0.7, 0.7], None]
    
    for exp in experiments:
        if os.path.exists(exp['folder']):
            obj_files = get_obj_files(exp['folder'])
            all_obj_files.append(obj_files)
        else:
            all_obj_files.append([])
    
    # 确定最大帧数
    max_len = max(len(files) for files in all_obj_files if files)
    
    # 采样
    target_frames = 50
    if max_len > target_frames:
        step = max_len // target_frames
        all_obj_files = [files[::step] if files else [] for files in all_obj_files]
        max_len = max(len(files) for files in all_obj_files if files)
    
    print(f"📹 生成对比GIF，共 {max_len} 帧\n")
    
    frames = []
    
    for i in range(max_len):
        print(f"   渲染进度: {i+1}/{max_len}", end='\r')
        
        images = []
        for obj_files, color in zip(all_obj_files, colors):
            if i < len(obj_files):
                epoch, obj_path = obj_files[i]
                try:
                    img = load_and_render_mesh(obj_path, renderer_comp, default_color=color)
                except:
                    img = np.ones((comparison_size, comparison_size, 3), dtype=np.uint8) * 128
            elif obj_files:
                epoch, obj_path = obj_files[-1]
                try:
                    img = load_and_render_mesh(obj_path, renderer_comp, default_color=color)
                except:
                    img = np.ones((comparison_size, comparison_size, 3), dtype=np.uint8) * 128
            else:
                img = np.ones((comparison_size, comparison_size, 3), dtype=np.uint8) * 128
            
            images.append(img)
        
        # 横向拼接
        combined = np.hstack(images)
        img_pil = Image.fromarray(combined)
        draw = ImageDraw.Draw(img_pil)
        
        try:
            font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", 20)
        except:
            font = ImageFont.load_default()
        
        titles = ['Baseline', 'Improved', 'Textured']
        for idx, title in enumerate(titles):
            x_pos = idx * comparison_size + comparison_size // 2 - 40
            draw.text((x_pos, 10), title, fill=(255, 255, 255), font=font, stroke_width=2, stroke_fill=(0, 0, 0))
        
        frames.append(np.array(img_pil))
    
    output_path = os.path.join(OUTPUT_DIR, '0_comparison.gif')
    print(f"\n   保存对比GIF...")
    imageio.mimsave(output_path, frames, fps=FPS, loop=0)
    print(f"✅ 对比GIF已保存: {output_path}\n")
else:
    print("\n⏭️  跳过对比GIF生成")

## 6. 完成

所有GIF已生成在 `gifs/` 文件夹中：
- `1_baseline_optimization.gif` - 基础版
- `2_improved_adam_optimization.gif` - 改进版
- `3_textured_v3_optimization.gif` - 最终版
- `0_comparison.gif` - 对比版（如果启用）

可以直接在左侧文件树中查看或下载！